# Longitudinal tracking simulations: playing with animations

Current tutors: S. Albright, H. Damerau, A. Lasheen, M. Taquet

Contributors: J. Flowerdew, L. Intelisano, D. Quartullo, F. Tecker, M. Zampetakis


## Links

- Introductory CAS website: https://indico.cern.ch/event/1622828/
- Programme of the CAS: https://indico.cern.ch/event/1622828/attachments/3196386/5990113/Timetable_Introductory2026_ver1.6b.pdf
- Python software installation for longitudinal exercises: https://github.com/cerncas/hands-on-longitudinal-exercises/blob/main/README.md
- Longitudinal hands-on, link to content and cheat sheets: https://indico.cern.ch/event/1622828/contributions/7202999/

## Introduction

This notebook is the companion of `LongitudinalHandsOnTracking`. It contains a complete tracking code: **there is nothing to program**, all the cells can be run as they are.

The goal is to observe the motion of a bunch of particles in the longitudinal phase space ($\phi$, $\Delta E$) with animations, and to analyze qualitatively the synchrotron motion **by changing the input parameters**: the position of the bunch in phase and energy, its length, its energy spread, the RF voltage...

1. Run the two cells of the first part once, they define the machine and the tracking functions.
2. Animate a bunch that is matched to the RF bucket.
3. In each of the following exercises, change one input parameter, run again, and explain what you observe.

Each animation takes a little while to compute. It can be replayed with the buttons below the figure.

The *Lecture slides* lines below point to the slides of the **Longitudinal Dynamics** lecture (F. Tecker) and of the **RF Systems** lecture (C. Völlinger) at this school, where the corresponding topics are introduced. The numbers are the slide numbers printed at the bottom of the slides.

## The machine and the tracking functions

The parameters are those of the scSPS at injection. The second cell contains the whole tracking code: the two equations of motion, applied turn after turn to all the particles of a bunch.

$$\phi_{n+1} = \phi_n + 2 \pi h \eta \frac{\Delta E_n}{\beta^2 E}$$
$$\Delta E_{n+1} = \Delta E_n + q V \sin (\phi_{n+1})$$

**Just run these two cells, then call a tutor!** You do not have to write this code, but we would like to go through it with you, so that you know what the computer is doing when you look at the animations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import c, e, m_p
from support_functions import generate_bunch, run_animation, separatrix

# The animations are embedded in the notebook, this line raises the default size limit
plt.rcParams["animation.embed_limit"] = 2**32

# Machine and beam parameters of the scSPS at injection

Ekin = 26e9  # eV
voltage = 15e6  # V

charge = 1  # in units of e
circumference = 6911.5  # m
harmonic = 4620
gamma_t = 18

# Energy and momentum, in eV and eV/c
E0 = m_p * c**2 / e
energy = Ekin + E0
momentum = np.sqrt(energy**2 - E0**2)
beta = momentum / energy
gamma = energy / E0

# Revolution and rf frequencies
t_rev = circumference / (beta * c)
f_rev = 1 / t_rev
f_rf = harmonic * f_rev
t_rf = 1 / f_rf

# Momentum compaction and phase slippage factor
alpha_c = 1 / gamma_t**2
eta = alpha_c - 1 / gamma**2

print("Kinetic energy: " + str(Ekin / 1e9) + " GeV")
print("Beta: " + str(beta))
print("Gamma: " + str(gamma))
print("Revolution period: " + str(t_rev * 1e6) + " mus")
print("RF frequency: " + str(f_rf / 1e6) + " MHz")
print("RF period: " + str(t_rf * 1e9) + " ns")
print("Momentum compaction factor: " + str(alpha_c))
print("Phase slippage factor: " + str(eta))

In [ ]:
# The tracking functions

# First equation of motion: the phase of the particles slips according to their energy offset


def drift(phaseInitial, energyInitial, harmonic, eta, beta, energy):
    newPhase = phaseInitial + 2 * np.pi * harmonic * eta * energyInitial / (beta**2 * energy)
    return newPhase


# Second equation of motion: the energy of the particles changes according to their phase in the cavity


def kick(energyInitial, phaseInitial, charge, voltage, acceleration=0):
    newEnergy = energyInitial + charge * voltage * np.sin(phaseInitial) - acceleration
    return newEnergy


# Generate a bunch, compute the rf bucket, and animate: drift then kick for all particles at each turn


def animate_bunch(
    bunch_position, bunch_length, energy_position, energy_spread, voltage, n_turns, figname
):
    # The bunch
    n_macroparticles = 10000
    phase_coordinates, energy_coordinates = generate_bunch(
        bunch_position, bunch_length, energy_position, energy_spread, n_macroparticles
    )

    # The separatrix of the rf bucket
    phase_array = np.linspace(0, 2 * np.pi, 1000)
    phase_sep, separatrix_array = separatrix(
        phase_array, f_rev, eta, beta, energy, charge, voltage, harmonic
    )

    # The energy spread that would match this bunch length (small amplitude approximation)
    synchrotronTune = np.sqrt(harmonic * eta * charge * voltage / (2 * np.pi * beta**2 * energy))
    matched_energy_spread = beta**2 * energy * synchrotronTune / (harmonic * eta) * bunch_length
    print("Synchrotron period: " + str(1 / synchrotronTune) + " turns")
    print(
        "Matched energy spread for this bunch length: " + str(matched_energy_spread / 1e6) + " MeV"
    )

    # The animation, one frame per turn: run_animation applies drift then kick at each turn
    framerate = 30  # frames per second
    return run_animation(
        phase_coordinates,
        energy_coordinates,
        drift,
        kick,
        [harmonic, eta, beta, energy],
        [charge, voltage],
        figname,
        n_turns,
        framerate,
        phase_sep=phase_sep,
        separatrix_array=separatrix_array,
    )

## Exercise 1: A bunch matched to the bucket

*Lecture slides: Tecker 36, 41, 59–60*

1. Run the cell below: it generates a bunch at the centre of the bucket and animates its motion for 100 turns.
2. What happens to the particles? To the bunch as a whole, and to its profiles in phase and in energy?
3. The function prints the energy spread that matches the bunch length. What happens if you change `bunch_length` and adapt `energy_spread` accordingly?

In [ ]:
animate_bunch(
    bunch_position=np.pi,  # rad, the centre of the bucket is at pi
    bunch_length=np.pi / 2,  # rad, full length
    energy_position=0,  # eV, with respect to the synchronous energy
    energy_spread=135e6,  # eV, full spread
    voltage=15e6,  # V
    n_turns=100,
    figname="Matched",
)


## Exercise 2: Phase error

*Lecture slides: Tecker 63, 65*

1. Inject the bunch with a phase error with respect to the bucket, e.g. a quarter of the bucket length.
2. How does the bunch move? How long does one oscillation take, compared to the synchrotron period printed by the function?

In [ ]:
# - copy the cell of Exercise 1 here and change bunch_position


## Exercise 3: Energy error

*Lecture slides: Tecker 65*

1. Put the bunch back at the centre of the bucket, and inject it with an energy error instead.
2. Compare with the phase error: could you tell them apart after a few turns?

In [ ]:
# - copy the cell of Exercise 1 here and change energy_position


## Exercise 4: Bunch length and energy spread mismatch

*Lecture slides: Tecker 61, 67*

1. Keep the bunch at the centre of the bucket, but make it too long for its energy spread (or too short).
2. How do the bunch length and the energy spread evolve? At which frequency?
3. When is the bunch the shortest? Could that be useful?

In [ ]:
# - copy the cell of Exercise 1 here and change bunch_length and energy_spread


## Exercise 5: Voltage mismatch

*Lecture slides: Tecker 62*

1. Take the matched bunch of Exercise 1 again, and change the RF voltage only.
2. What do you observe, and why?

In [ ]:
# - copy the cell of Exercise 1 here and change voltage


## Exercise 6: After many synchrotron periods

*Lecture slides: Tecker 61–63*

1. Take one of the mismatched cases above and track it for many synchrotron periods, e.g. 600 turns.
2. What happens to the bunch, and to its profiles? Is that reversible?

In [ ]:
# - copy the cell of Exercise 1 here with one of your mismatches, and increase n_turns
